In [5]:
!mkdir yolov1

In [9]:
%%writefile yolov1/dataset.py
import torch
import os
from PIL import Image
import pandas as pd
"""
this returns image together with their bounding boxes
1. first takes annotations which contains the label like 0.0001xml
2. also take annotations which contains the image filename e.g 0.0001jpg
3. label_path(inside it contains everything i.e (class_label, x,y,w,h) which shows where exactly the position(center) of bounding box
4. we have already have the coordinates of the center of the bounding boxes, now we convert them to match the grid (SxS) i.e how they can be pos--
itioned relative to each cells presented
"""
class VOCDATASET(torch.utils.data.Dataset):
    def __init__(self, csv_file, image_dir, label_dir, S = 7, B = 2, C = 20, transform = None):
        self.annotations = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transforms = transform
        self.S = S
        self.B = B
        self.C = C
    def __len__(self):
        return len(self.annotations)
    def __getitem__(self, index):
        label_path = os.path.join([self.label_dir, self.annotations.iloc[index, 1]])
        boxes = [] 
        with open(label_path) as f:
            for lines in f.readlines():
                class_label, x,y,w,h = [float(value) if float(value) != int(float(value)) else int(value) for value in lines.replace('\n', '').split()]
                boxes.append([class_label, x, y, w, h])
        image_path = os.path.join([self.image_dir, self.annotations.iloc[index, 0]]) 
        img = Image.open(imag_path)
        boxes = torch.tensor(boxes)

        if self.transforms:
            img, boxes = self.transforms(img, boxes)

        label_matrix = torch.zeros(self.S, self.S, self.B*5 + self.C)
        for box in boxes:
            class_label, x, y, w, h = box.list()
            class_label = int(class_label)
            """
            i,j = 0,0 for every grid 
            when we calculate the below it will give the exact position of the center according to grid e.g i,j = 5,6 represents that the
            center is in row 5 and column 6
            """
            i,j = (self.S * y), (self.S * x)

            #now to calculate the X_cell and y_cell (coordinates the bounding boxes)
            x_cell, y_cell = self.S * x - j, self.S*y - i

            #now for the width of the box
            width_cell, height_cell = self.S * w, self.S * h
            """
            this is for the below conditionality
            1. we know that i and j contain the center of the object
            2. so it is responsible for the prediction of the bounding boxes
            3. provided that row, column, confidence_scores is equal to zero it means that we did not assign as it is supposed to be 1 since we know 
            i,j is the center to which the object in the grid is located
            4. 
            """
            if label_matrix[i,j, 20] == 0:
                
                label_matrix[i,j, 20] = 1

                box_coordinates = torch.tensor([x_cell, y_cell, width_cell, height_cell])

                label_matrix[i,j, 21:25] = box_coordinates

                label_matrix[i,j, class_label] = 1
        return img, label_matrix
                
##understood           
        

Overwriting yolov1/dataset.py


In [11]:
%%writefile yolov1/loss.py
#loss function implementation
import torch
from torch import nn
from utils import intersection_over_union
class YoloLoss(nn.Module):
    def __init__(self, S = 7, B = 2, C = 20):
        super().__init__()
        self.mse = nn.MSE(reduction = 'sum')
        self.S = S
        self.B = B
        self.C = C

        self.noobj = 0.5
        self.obj = 5
    def forward(self, predictions, target):
        iou_b1 = intersection_over_union(predictions[..., 21:25], target[..., 21:25])
        iou_b2 = intersection_over_union(predictions[..., 26:30], target[..., 21:25])
        ious = torch.cat([iou_b1.unsqueeze(0), iou_b2.unsqueeze(0)], dim = 0)

        iou_max, best_box = ious.max(dim = 0)
        # this returns (0,1) as indices 
        box_exists = target[..., 20].unsqueeze(3) #this returns 1obj in the paper of shape (batch_size, 7,7,1)
        
        #box_loss
        box_predictions = (
            box_exists * (best_box*predictions[..., 26:30]) # this is the same as (batch_size,7,7,1) x (batch_size,7,7,4)
            + ((1-best_box) * predictions[..., 21:25])
        )
        box_targets = box_exists * target[..., 21:25]
        

        box_predictions[..., 2:4] = torch.sign(box_predictions[...,2:4]*torch.sqrt(torch.abs(box_predictions[..., 2:4]+1e-6))) # same as (batch_size,7,7,2)
        box_targets[..., 2:4] = torch.sqrt(box_targets[...,2:4])

        box_loss = self.mse(
            nn.Flatten(box_predictions, end_dim=-2),#(batch_size*7*7,4)
            nn.Flatten(box_targets, end_dim=-2)
        )
        #object_loss or confidence score loss termed as Ci in the paper
        pred_box = (
            best_box*predictions[..., 25:26]+(1-best_box)*predictions[..., 20:21] # (batch_size, 7,7,1)
        )
        object_loss = self.mse(
            nn.Flatten(pred_box),# same as (batch_size*7*7*1)
            nn.Flatten(box_targets[20:21])
        )
        #no object loss
        #for the first bounding box
        no_object_loss = self.mse(
            torch.flatten((1 - box_exists) * predictions[..., 20:21], start_dim=1), #(batch_size, 7*7*1)
            torch.flatten((1 - box_exists) * target[..., 20:21], start_dim=1),
        #for the second bounding box
        no_object_loss += self.mse(
            torch.flatten((1-box_exists)*predictions[...,25:26],start_dim = 1),#(batch_size, 7*7*1)
            torch.flatten((1-box_exists)*target[...,20:21], start_dim = 1)
        )
        #class loss
        class_loss = self.mse(
            torch.flatten(box_exists * predictions[..., :20], end_dim = -2)#(batch_size*7*7,1)
            torch.flatten(box_exists * target[..., :20], end_dim = -2)
        )

        loss = (
            self.obj*box_loss + self.obj*object_loss
            + self.noobj*no_object_loss + class_loss
        )
        return loss


Writing yolov1/loss.py


In [13]:
%%writefile yolov1/model.py
import torch
from torch import nn
"""
(kernel_size, outputfeatures, stride, padding)
'M' represents MaxPoolLayer
padding = (Kernel_size -1)/2
"""
architecture_config = [
    (7, 64, 2, 3),
    'M',
    (3, 192,1, 1),
    'M',
    (1,128, 1,0),
    (3, 256,1,1),
    (1,256,1,0),
    (3,512,1,1),
    'M',
    [(1,256,1,0), (3,512,1,1), 4],
    (1,512,1,0),
    (3,1024,1,1),
    'M',
    [(1,512,1,0), (3,1024,1,1), 2],
    (3,1024,1,1),
    (3,1024,2,1),
    (3,1024,1,1),
    (3,1024,1,1)
]
class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, **kwargs):
        super().__init__()
        self.in_channels = in_channels
        self.batch_norm = nn.BatchNorm2d(out_channels)
        self.leaky_relu = nn.LeakyReLU(0.1)
        self.CNN = nn.Conv2d(self.in_channels, out_channels, bias= False, **kwargs)
    def forward(self, x):
        x = self.leaky_relu(self.batch_norm(self.CNN(x)))
        return x

class Yolov1(nn.Module):
    def __init__(self, in_channels=3, **kwargs):
        super().__init__()
        self.architecture_config = architecture_config
        self.in_channels = in_channels
        self.darknet = self._conv_layers(self.architecture_config)
        self.fcs = self._conn_layers(**kwargs)
        
    def _conv_layers(self, architecture):
        in_channels = self.in_channels
        layers = []
        for x in architecture:
            if type(x) == tuple:
                layers +=[
                    CNNBlock(
                        in_channels,
                        x[1],
                        kernel_size = x[0],
                        padding = x[3]
                    )
                ]
                in_channels = x[1]
            elif type(x) == str:
                layers +=[
                    nn.MaxPool2d(2)
                ]
            elif type(x) == list:
                conv1 = x[0]
                conv2 = x[1]
                repeat_num = x[2]
                for _ in range(repeat_num):
                    layers +=[
                        CNNBlock(
                        in_channels,
                        conv1[1],
                        kernel_size = conv1[0],
                        padding = conv1[3]
                    )
                    ]
                    in_channels = conv1[1]
                    layers += [
                        CNNBlock(
                        in_channels,
                        conv2[1],
                        kernel_size = conv2[0],
                        padding = conv2[3]
                    )
                    ]
                    in_channels = conv2[1]
        return nn.Sequential(*layers)
    def _conn_layers(self, split_size=7, class_size=20, num_boxes=2):
        S = split_size
        B = num_boxes
        C = class_size
        return nn.Sequential(
            nn.Flatten(),
            nn.Linear(S*S*1024, 4096),
            nn.Dropout(0.0),
            nn.LeakyReLU(0.1),
            nn.Linear(4096, S*S*(C + (B*5)))
        )
    def forward(self, x):
        x = self.darknet(x)
        x = self.fcs(x)
        return x

Writing yolov1/model.py


In [17]:
%%writefile yolov1/train.py
import torch
from torch import nn
from torch.utils.data import DataLoader
from dataset import VOCDATASET
import torchvision.transforms as transforms
import torchvision.transforms.functional as F
from model import Yolov1
from utils import (
    non_max_suppression,
    mean_average_precision,
    intersection_over_union,
    cellboxes_to_boxes,
    get_bboxes,
    plot_image,
    save_checkpoint,
    load_checkpoint,
)
from loss import YoloLoss
from tqdm.auto import tqdm

seed = 123
torch.manual_seed(seed)

# Hyperparameters etc. 
LEARNING_RATE = 2e-5
DEVICE = "cuda" if torch.cuda.is_available else "cpu"
BATCH_SIZE = 16 
WEIGHT_DECAY = 0
EPOCHS = 1000
NUM_WORKERS = 2
PIN_MEMORY = True
LOAD_MODEL = False
LOAD_MODEL_FILE = "overfit.pth.tar"
IMG_DIR = "data/images"
LABEL_DIR = "data/labels"


class Compose(object):
    def __init__(self, transforms):
        self.transforms = transforms #takes in a list
    def __call__(self,img, bboxes):
        for t in self.transforms:
            img, bboxes = t(img), bboxes

        return img, bboxes
transform = Compose([transforms.Resize((448, 448)), transforms.ToTensor(),])

def train_fn(loss_fn, dataloader, model, optimizer, device):
    loop = tqdm(dataloader, leave = True)
    mean_list = []

    for batch, (x,y) in enumerate(loop):
        x,y = x.to(device),y.to(device)
        out = model(x)
        loss = loss_fn(out, y)
        mean_list.append(loss)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loop.set_postfix(loss = loss.item())

    print(f'Mean was {sum(mean_list)/len(mean_list)}')

def main():
    model = Yolov1()
    loss_fn = YoloLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr = LEARNING_RATE, weight_decay = WEIGHT_DECAY)
    if LOAD_MODEL:
        load_checkpoint(torch.load(LOAD_MODEL_FILE),model, optimizer)
    train_dataset = VOCDataset(
        "data/100examples.csv",
        transform=transform,
        img_dir=IMG_DIR,
        label_dir=LABEL_DIR,
    )

    test_dataset = VOCDataset(
        "data/test.csv", transform=transform, img_dir=IMG_DIR, label_dir=LABEL_DIR,
    )

    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        shuffle=True,
        drop_last=True,
    )

    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        shuffle=True,
        drop_last=True,
    )
    for epoch in range(epochs):
        pred_boxes, target_boxes = getbboxes()

        mean_average_precision = mean_average_precision(pred_boxes, target_boxes)
        print(f'Mean average precision is {mean_average_precision}')

        train_fn(train_loader, model, loss_fn, optimizer, device = DEVICE)

if __name__ ==' __main__':
    main()

Writing yolov1/train.py


In [ ]:
#getting data
!wget 'https where you download your whole dataset'#<= this should be normal not a string

#how to extract tar files
tar xf 'https....voc_label.py'#<= this is not a string
#this does -> returns yolo.txt in a format of [class_id x_center y_center width height]
python voc_label.py

cat 2007_train.txt 2007_val.txt 2012_*.txt > train.txt # the > = redirect output to a file(meaning)
cp 2007_test.txt test.txt

mkdir old_txt_files
mv 2007* 2012* old_txt_files/       

mkdir data
mkdir data/images
mkdir data/labels

mv VOCdevkit/VOC2007/JPEGImages/*.jpg data/images/                                      
mv VOCdevkit/VOC2012/JPEGImages/*.jpg data/images/                                      
mv VOCdevkit/VOC2007/labels/*.txt data/labels/                                          
mv VOCdevkit/VOC2012/labels/*.txt data/labels/ 

rm -rf VOCdevkit/
mv test.txt old_txt_files/
mv train.txt old_txt_files/ 

In [ ]:
import os
import csv
read_train = open('train.txt', 'r').readline()
with open('train.csv', 'w', newline = '') as train_file:
    for line in read_train:
        image = line.split('/')[-1].replace('\n', '')
        label = image.replace('jpg', 'txt')
        result = [image, label]
        writer = csv.writer(train_file)
        writer.writerow(result)

read_test = open('test.txt', 'r').readline()
with open('test.csv', 'w', newline = '') as test_file:
    for line in read_test:
        image = line.split('/')[-1].replace('\n', '')
        label = image.replace('jpg','txt')
        data = [image, label]
        writer = csv.writer(test_file)
        writer.writerow(data)
    

In [1]:
def intersection_over_union(target, predictions, box_format = 'mid-point'):
    """
    given that midpoint represent [class_label, conf_score, x, y, w, h]
    given that corners represent [...x1, y1, x2, y2]
    """
    if box_format == 'mid-point':
        box1_x1 = predictions[..., 21:22] - predictions[..., 23:24]/2
        box1_y1 = predictions[..., 22:23] - predictions[..., 24:25]/2
        box1_x2 = predictions[..., 21:22] + predictions[..., 23:24]/2
        box1_y2 = predictions[..., 22:23] + predictions[..., 23:24]/2
        
        box2_x1 = target[..., 26:27] - target[..., 28:29]/2
        box2_y1 = target[..., 27:28] - target[..., 29:30]/2
        box2_x2 = target[..., 26:27] + target[..., 28:29]/2
        box2_y2 = target[..., 27:28] + target[..., 29:30]/2
    if box_format == 'corners':
        box1_x1 = predictions[..., 21:22]
        box1_y1 = predictions[..., 22:23]
        box1_x2 = predictions[..., 23:24]
        box1_y2 = predictions[..., 24:25]
        
        box1_x1 = target[..., 21:22]
        box1_y1 = target[..., 22:23]
        box1_x2 = target[..., 23:24]
        box1_y2 = target[..., 24:25]

    x1 = torch.max(box1_x1, box2_x1)
    y1 = torch.max(box1_y1, box2_y1)
    x2 = torch.min(box1_x2, box2_x2)
    y2 = torch.min(box1_y2, box2_y2)

    intersection = ((x2-x1).clamp(0)) * ((y2-y1).clamp(0))
    area_b1 = abs((box1_x1 - box1_x2) * (box1_y1 - box1_y2))
    area_b2 = abs((box2_x1 - box2_x2) * (box2_y1 - box2_y2))

    iou = intersection / (area_b1 + area_b2 - intersection + 1e-6)
    
    return iou

        

In [ ]:
def non_max_suppresion(bboxes, threshold, iou_threshold, box_format = 'corners'):
    assert type(bboxes) == list
    bboxes = [box for box in bboxes if box[1]>threshold ]

    bboxes = sorted(bboxes, key = lambda x: x[1], reverse = True)
    bboxes_after_nms = []
    while bboxes:
        chosen_box = bboxes.pop(0)

        bboxes = [
            box
            for box in bboxes
            if chosen_box[0] != box[0] or
            intersection_over_union(
                torch.tensor(box[2:]),
                torch.tensor(chosen_box[2:])
                box_format = box_format
            )
        ]
        bboxes_after_nms.append(chosen_box)
    return bboxes_after_nms

In [ ]:
def get_bboxes(loader, model, threshold, iou_threshold, box_format = 'mid-point', pred_format = 'cells', device = 'cuda'):
    
    all_pred_boxes = []
    all_true_boxes = []
    model.eval()
    train_idx = 0
    for batch_idx, (x, labels) in enumerate(dataloader):
        x = x.to(device)
        labels = labels.to(device)

        with torch.no_grad():
            predictions = model(x)

        batch_size = x[0]
        pred_boxes = cellboxes_to_boxes(predictions)
        true_boxes = cellboxes_to_boxes(labels)

        for idx in range(batch_size):
            nms_boxes = non_max_suppression(
                pred_boxes[idx],
                threshold,
                iou_threshold,
                box_format
            )
            for nms_box in nms_boxes:
                all_pred_boxes.append([train_idx] + nms_box)

            for true_box in true_boxes[idx]:
                all_true_boxes.append([train_idx] + true_boxes)

            train_idx += 1
    model.train()
    return all_pred_boxes, all_true_boxes

In [ ]:
def cellboxes_to_boxes(out, S=7): # convert predictions into a list of all predictions e.g [[class_label, pred_score,x,y,w,h],[class_label, pred_score,x,y,w,h]...]
    converted_pred = converted_pred(out.shape[0], S*S, -1)
    converted_pred[..., 0] = converted_pred[..., 0].long()
    all_boxes = []
    for ex_id in range(out.shape[0]):
        boxes = []
        for box_id in range(S*S):
            boxes.append([x.item() for x in converted_pred[ex_id, box_id, :]])
        all_boxes.append(boxes)
    return all_boxes

In [ ]:
def convert_cellboxes(predictions, S = 7):
    predictions = predictions.to('cpu')
    batch_size = predictions.shape[0]
    predictions = predictions.reshape(batch_size, S, S, -1)
    bboxes1 = predictions[..., 21:25]
    bboxes2 = predictions[..., 26:30]
    scores = torch.cat(predictions[..., 20].unsqueeze(0), predictions[..., 25].unsqueeze(0), dim = 0)
    best_box = torch.argmax(scores, dim = 0)
    best_predictions = best_box* bboxes1 + ((1-best_box)*bboxes2)
    cell_indices = torch.arange(7).repeat(batch_size, 7, 1).unsqueeze(-1)
    x = 1/S * (best_predictions[...,:1] + cell_indices)
    y = 1/S * (best_predictions[...,1:2] + cell_indices.permute(0, 2, 1, 3))
    w_y = 1/S * (best_predictions[...,2:4])
    converted_pred = torch.cat(x,y,w_y, dim = -1)
    predicted_class = best_prediction[..., :20].argmax(-1).unsqueeze(-1)
    confidence_score = torch.max(best_prediction[..., 20].unsqueeze(-1), best_prediction[..., 25].unsqueeze(-1))
    converted_pred = torch.cat((predicted_class, confidence_score, converted_pred) dim = -1)
    return converted_pred
    

In [ ]:
def load_checkpoint(checkpoint, model, optimizer):
    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['optimizer'])

def save_model(state, filename = '...pth.tar'):
    torch.save(state, filename)

In [ ]:
def plot_image(image, bboxes):
    img = nparray(image)
    height, width,_ = im.shape

    fig, axs = plt.subplots(1)

    axs.imshow(img)

    for box in bboxes:
        box = box[2:]
        assert len(box) ==4 ,'must be a value of 4'
        upper_left_x = box[0] - box[2]/2 
        upper_left_y = box[1] - box[3]/2
        rect = patches.Rectangle(
            (upper_left_x * width, upper_left_y * height),
            box[2] * width,
            box[3] * height,
            linewidth = 1,
            edgecolor = 'r',
            facecolor = 'none'  
        )
        axs.add_patch(rect)
    plt.show()
        

In [ ]:
def mean_average_precision(pred_box, true_box, box_format = 'mid-point', iou_threshold = 0.5, num_classes = 20):
    average_precision = []
    
    epsilon = 1e-3
    ground_truths = []
    detections = []

    for c in range(num_classes):
        for box in pred_box:
            if box[1] == c:
                detections.append(box)
        for box in true_box:
            if box[1] == c:
                ground_truths.append(box)

        amount_bboxes = Counter(ground_truths)
        for key, value in amount_bboxes.items():
            amount_bboxes[key] = torch.zeros(val)

        TP = len(detections)
        FP = len(detections)
        num_gts = len(ground_truths)

        if len(ground_truths) == 0:
            continue
            
        for detection_idx, detection in enumerate(detections):
            ground_truth_image = [
                box
                for box in ground_truths
                if box[0] == detection[0]
            ]
            num_gt_image = len(ground_truth_image)
            best_iou = 0
            for idx, gt in enumerate(ground_truth_image):
                iou = intersection_over_union(
                    gt[3:],
                    detection[3:],
                    box_format = box_format,
                )
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = idx
            if best_iou > iou_threshold:
                if amount_bboxes[detection[0]][best_gt_idx] == 0:
                    amount_bboxes[detection[0]][best_gt_idx] = 1
                    TP[detection_idx] = 1
                else:
                    FP[detection_idx] = 1
            else:
                FP[detection_idx] = 1
        TP_cumsum = torch.cumsum(TP, dim= 0)
        FP_cumsum = torch.cumsum(FP, dim = 0)
        recall =TP_cumsum / (num_gt + epsilon)
        precision = TP_cumsum/(TP_cumsum + FP_cumsum + epsilon)
        precision = torch.cat(torch.tensor([0]), precision)
        recall = torch.cat(torch.tensor([1]), recall)
        average_precision.append(torch.trapz(precision, recall))

    return (sum(average_precision)/ len(average_precision))
        
        

        